In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

# 官方示例 1

LangChain 教程：[Step Back Prompting](https://python.langchain.com/v0.1/docs/use_cases/query_analysis/techniques/step_back/)

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system = """你擅长从特定问题中提取出更一般的问题，从而获得回答特定问题所需的基本原理。

你将被问到关于一组用于构建LLM驱动应用程序的软件，包括LangChain、LangGraph、LangServe和LangSmith。

LangChain是一个Python框架，提供了大量的集成，可以轻松组合以构建LLM应用程序。
LangGraph是一个建立在LangChain之上的Python包，使构建有状态的、多角色的LLM应用程序变得容易。
LangServe是一个建立在LangChain之上的Python包，使将LangChain应用程序部署为REST API变得容易。
LangSmith是一个平台，使追踪和测试LLM应用程序变得容易。

针对用户关于这些产品中一个或多个的特定问题，写出一个更一般的问题，以便回答特定问题。

如果你不认识一个词或缩写词，不要试图重写它。

写简洁的问题。"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
step_back = prompt | llm | StrOutputParser()

In [3]:
question = (
    "我使用Gemini Pro和像vectorstores和duckduckgo搜索这样的工具构建了一个LangGraph代理。"
    "如何从事件流中获取仅LLM的调用？"
)
result = step_back.invoke({"question": question})
print(result)

如何在LLM应用程序中从事件流中提取特定类型的调用？


In [4]:
from langchain_core.runnables import RunnablePassthrough

step_back_and_original = RunnablePassthrough.assign(step_back=step_back)

step_back_and_original.invoke({"question": question})

{'question': '我使用Gemini Pro和像vectorstores和duckduckgo搜索这样的工具构建了一个LangGraph代理。如何从事件流中获取仅LLM的调用？',
 'step_back': '如何在LangGraph中从事件流中提取特定类型的调用？'}

# 官方示例 2

LangChain  Cookbook：[Step-Back Prompting (Question-Answering)](https://github.com/langchain-ai/langchain/blob/master/cookbook/stepback-qa.ipynb)

In [5]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "The Police的成员可以合法逮捕人吗？",
        "output": "The Police的成员能做什么？",
    },
    {
        "input": "Jan Sindel是在哪个国家出生的？",
        "output": "Jan Sindel的个人历史是什么？",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """你是世界知识的专家。你的任务是后退一步，将问题改写为更通用的后退问题，这样更容易回答。这里有几个例子：""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

In [6]:
few_shot_prompt

FewShotChatMessagePromptTemplate(examples=[{'input': 'The Police的成员可以合法逮捕人吗？', 'output': 'The Police的成员能做什么？'}, {'input': 'Jan Sindel是在哪个国家出生的？', 'output': 'Jan Sindel的个人历史是什么？'}], example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['output'], template='{output}'))]))

In [7]:
prompt

ChatPromptTemplate(input_variables=['question'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='你是世界知识的专家。你的任务是后退一步，将问题改写为更通用的后退问题，这样更容易回答。这里有几个例子：')), FewShotChatMessagePromptTemplate(examples=[{'input': 'The Police的成员可以合法逮捕人吗？', 'output': 'The Police的成员能做什么？'}, {'input': 'Jan Sindel是在哪个国家出生的？', 'output': 'Jan Sindel的个人历史是什么？'}], example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['output'], template='{output}'))])), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], template='{question}'))])

In [8]:
step_back = prompt | ChatOpenAI(temperature=0) | StrOutputParser()

In [9]:
question = "LLM智能体的任务分解是什么？"
step_back.invoke({"question": question})

'LLM智能体的任务如何分解？'

# 优化Naive-RAG

## 索引

In [10]:
from dotenv import load_dotenv
from langchain import hub
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载环境变量
load_dotenv()

# Load
file_path = "./data/translated_article.txt"
loader = TextLoader(file_path, encoding='utf-8')
docs = loader.load()

# Split
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
markdown_text = "\n\n".join([doc.page_content for doc in docs])

md_header_splits = markdown_splitter.split_text(markdown_text)

chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

# Embed
embedding = OpenAIEmbeddings()

# Store
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

# Retrieve
retriever = vectorstore.as_retriever()

## Step-Back

In [11]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "The Police的成员可以合法逮捕人吗？",
        "output": "The Police的成员能做什么？",
    },
    {
        "input": "Jan Sindel是在哪个国家出生的？",
        "output": "Jan Sindel的个人历史是什么？",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """你是世界知识的专家。你的任务是后退一步，将问题改写为更通用的后退问题，这样更容易回答。这里有几个例子：""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

step_back = prompt | ChatOpenAI(temperature=0) | StrOutputParser()

## 测试子链

In [12]:
# test
step_back.invoke({"question": question})

'LLM智能体的任务如何分解？'

In [13]:
step_back.invoke(question)

'LLM智能体的任务如何分解？'

In [15]:
# 后退问题检索到的文档列表
(step_back | retriever).invoke({"question": question})

[Document(page_content='文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```  \n许多研究尝试了如何构建上下文示例以最大化性能，并发现**提示格式、训练示例及其顺序的选择可以极大地影响模型的性能**，从几乎是随机猜测到接近最优状态的表现。  \n[Zhao等人（2021年）](https://arxiv.org/abs/2102.09690)研究了少样本分类，并发现几种与LLM（他们使用的是GPT-3）相关的偏见导致了性能的高变异性：（1）*多数标签偏见*，如果示例中的标签分布不平衡；（2）*近期偏见*，模型可能倾向于重复最后出现的标签；（3）*常见Token偏见*，LLM倾向于生成常见的Token而不是罕见的Token。为了克服这些偏见，他们提出了一个方法，即当输入字符串为`N/A`时，调整模型输出的标签概率为均匀。', metadata={'Header 1': '基础提示', 'Header 2': '少样本'}),
 Document(page_content='# 提示工程  \n日期：2023年3月15日 | 预计阅读时间：21分钟 | 作者：Lilian Weng  \n**提示工程**（Prompt Engineering），亦称**上下文提示**（In-Context Prompting），涉及如何与大语言模型（LLM）进行交流，通过不更新模型权重的方式引导其行为，以获得预期的结果。这是一门基于实验的科学，不同模型之间提示工程的效果可能大相径庭，因此需要大量的试验和启发式方法。  \n本文专注于自回归语言模型的提示工程，不包括完形填空测试、图像生成或多模态模型。提示工程的核心目标是实现模型的对齐和可控制性。详情可参阅我之前关于可控文本生成的[文章](https://lilianweng.github.io/posts/2021-01-02-controllable-text-generation/)。', metadata={'Header 1': '提示工程'}),
 Document(page_content='## 少样本  \n**少样本学习**通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。因此，少样本学习通常比零样本学习有更好的表现。但这种方法

In [18]:
# 后退问题检索到的文档列表
(step_back | retriever).invoke(question)

[Document(page_content='文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```  \n许多研究尝试了如何构建上下文示例以最大化性能，并发现**提示格式、训练示例及其顺序的选择可以极大地影响模型的性能**，从几乎是随机猜测到接近最优状态的表现。  \n[Zhao等人（2021年）](https://arxiv.org/abs/2102.09690)研究了少样本分类，并发现几种与LLM（他们使用的是GPT-3）相关的偏见导致了性能的高变异性：（1）*多数标签偏见*，如果示例中的标签分布不平衡；（2）*近期偏见*，模型可能倾向于重复最后出现的标签；（3）*常见Token偏见*，LLM倾向于生成常见的Token而不是罕见的Token。为了克服这些偏见，他们提出了一个方法，即当输入字符串为`N/A`时，调整模型输出的标签概率为均匀。', metadata={'Header 1': '基础提示', 'Header 2': '少样本'}),
 Document(page_content='## 少样本  \n**少样本学习**通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。因此，少样本学习通常比零样本学习有更好的表现。但这种方法会消耗更多的Token，并且当输入和输出文本较长时可能触及上下文长度的限制。  \n```\n文本: （劳伦斯在舞台上）到处跳跃，跳舞，跑步，出汗，擦脸，充分展现了他初次成名时的惊人才华。\n情感: 正面\n\n文本: 尽管所有迹象都表明相反，这部糟糕的电影还是设法冒充了一部正式的商业影片，收取全价门票，在电视上大肆宣传，自称能够娱乐小孩和成人。\n情感: 负面\n\n文本: 多年来第一次，德尼罗在情感上有了深刻的挖掘，可能是因为受到了合作伙伴出色表现的启发。\n情感: 正面', metadata={'Header 1': '基础提示', 'Header 2': '少样本'}),
 Document(page_content='# 提示工程  \n日期：2023年3月15日 | 预计阅读时间：21分钟 | 作者：Lilian Weng  \n**提示工程**（Prompt Engineering），亦称**上下文提示**（In-Context Prompting），涉

In [ ]:
# 原始问题检索到的文档列表

In [17]:
retriever.invoke(question)

[Document(page_content='文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```  \n许多研究尝试了如何构建上下文示例以最大化性能，并发现**提示格式、训练示例及其顺序的选择可以极大地影响模型的性能**，从几乎是随机猜测到接近最优状态的表现。  \n[Zhao等人（2021年）](https://arxiv.org/abs/2102.09690)研究了少样本分类，并发现几种与LLM（他们使用的是GPT-3）相关的偏见导致了性能的高变异性：（1）*多数标签偏见*，如果示例中的标签分布不平衡；（2）*近期偏见*，模型可能倾向于重复最后出现的标签；（3）*常见Token偏见*，LLM倾向于生成常见的Token而不是罕见的Token。为了克服这些偏见，他们提出了一个方法，即当输入字符串为`N/A`时，调整模型输出的标签概率为均匀。', metadata={'Header 1': '基础提示', 'Header 2': '少样本'}),
 Document(page_content='# 提示工程  \n日期：2023年3月15日 | 预计阅读时间：21分钟 | 作者：Lilian Weng  \n**提示工程**（Prompt Engineering），亦称**上下文提示**（In-Context Prompting），涉及如何与大语言模型（LLM）进行交流，通过不更新模型权重的方式引导其行为，以获得预期的结果。这是一门基于实验的科学，不同模型之间提示工程的效果可能大相径庭，因此需要大量的试验和启发式方法。  \n本文专注于自回归语言模型的提示工程，不包括完形填空测试、图像生成或多模态模型。提示工程的核心目标是实现模型的对齐和可控制性。详情可参阅我之前关于可控文本生成的[文章](https://lilianweng.github.io/posts/2021-01-02-controllable-text-generation/)。', metadata={'Header 1': '提示工程'}),
 Document(page_content='## 少样本  \n**少样本学习**通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。因此，少样本学习通常比零样本学习有更好的表现。但这种方法

## 生成最终输出

### test 1

In [19]:
# Generate
# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{normal_context}"
    "{step_back_context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI()
output_parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "normal_context": retriever | format_docs, 
        "step_back_context": step_back | retriever | format_docs, 
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | output_parser
)

question = "什么是few shot？"
print(f"Q: {question}\nA: ")

result = rag_chain.invoke(question)
print(result)


Q: 什么是few shot？
A: 
少样本（Few-shot）学习是一种模型提示方法，通过提供少量高质量的示例来帮助模型更好地理解人类的意图和评价标准。这些示例包含目标任务的输入和期望输出，通常比零样本学习效果更好。然而，这种方法可能会消耗更多Token，并且在处理长文本时可能会触及上下文长度的限制。


### test 2

In [20]:
# Generate
# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{normal_context}"
    "{step_back_context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI()
output_parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "normal_context": retriever | format_docs, 
        "step_back_context": step_back | retriever | format_docs, 
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | output_parser
)

question = "什么是few shot？"
print(f"Q: {question}\nA: ")

result = rag_chain.invoke({"question": question})
print(result)


Q: 什么是few shot？
A: 


TypeError: argument 'text': 'dict' object cannot be converted to 'PyString'

### test 3

In [21]:
# Generate
# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{normal_context}"
    "{step_back_context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI()
output_parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "normal_context": (lambda x: x["question"])| retriever | format_docs, 
        "step_back_context": step_back | retriever | format_docs, 
        "question": lambda x: x["question"]
    }
    | prompt
    | llm
    | output_parser
)

question = "什么是few shot？"
print(f"Q: {question}\nA: ")

result = rag_chain.invoke({"question": question})
print(result)


Q: 什么是few shot？
A: 
少样本学习（Few-shot learning）是一种模型提示方法，通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，来帮助模型更好地理解人类的意图和评价标准，从而在新任务上表现更好。


In [22]:
(lambda x: x["question"]).invoke({"question": question})

AttributeError: 'function' object has no attribute 'invoke'

In [23]:
type((lambda x: x["question"]))

function

In [24]:
from langchain_core.runnables import RunnableLambda

In [25]:
(RunnableLambda(lambda x: x["question"])).invoke({"question": question})

'什么是few shot？'

In [26]:
(RunnableLambda(lambda x: x["question"]) | retriever).invoke({"question": question})

[Document(page_content='# 基础提示  \n零样本（Zero-shot）和少样本（Few-shot）学习是两种基本的模型提示方法，这两种方法最早由多篇大语言模型论文提出，并广泛用于评估大语言模型的性能。', metadata={'Header 1': '基础提示'}),
 Document(page_content='## 少样本  \n**少样本学习**通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。因此，少样本学习通常比零样本学习有更好的表现。但这种方法会消耗更多的Token，并且当输入和输出文本较长时可能触及上下文长度的限制。  \n```\n文本: （劳伦斯在舞台上）到处跳跃，跳舞，跑步，出汗，擦脸，充分展现了他初次成名时的惊人才华。\n情感: 正面\n\n文本: 尽管所有迹象都表明相反，这部糟糕的电影还是设法冒充了一部正式的商业影片，收取全价门票，在电视上大肆宣传，自称能够娱乐小孩和成人。\n情感: 负面\n\n文本: 多年来第一次，德尼罗在情感上有了深刻的挖掘，可能是因为受到了合作伙伴出色表现的启发。\n情感: 正面', metadata={'Header 1': '基础提示', 'Header 2': '少样本'}),
 Document(page_content='文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```  \n许多研究尝试了如何构建上下文示例以最大化性能，并发现**提示格式、训练示例及其顺序的选择可以极大地影响模型的性能**，从几乎是随机猜测到接近最优状态的表现。  \n[Zhao等人（2021年）](https://arxiv.org/abs/2102.09690)研究了少样本分类，并发现几种与LLM（他们使用的是GPT-3）相关的偏见导致了性能的高变异性：（1）*多数标签偏见*，如果示例中的标签分布不平衡；（2）*近期偏见*，模型可能倾向于重复最后出现的标签；（3）*常见Token偏见*，LLM倾向于生成常见的Token而不是罕见的Token。为了克服这些偏见，他们提出了一个方法，即当输入字符串为`N/A`时，调整模型输出的标签概率为均匀。', metadata={'Header 1': '基础提示', 'Header 2': '少样本'})

In [27]:
(RunnableLambda(lambda x: x["question"]) | retriever | format_docs).invoke({"question": question})

'# 基础提示  \n零样本（Zero-shot）和少样本（Few-shot）学习是两种基本的模型提示方法，这两种方法最早由多篇大语言模型论文提出，并广泛用于评估大语言模型的性能。\n\n## 少样本  \n**少样本学习**通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。因此，少样本学习通常比零样本学习有更好的表现。但这种方法会消耗更多的Token，并且当输入和输出文本较长时可能触及上下文长度的限制。  \n```\n文本: （劳伦斯在舞台上）到处跳跃，跳舞，跑步，出汗，擦脸，充分展现了他初次成名时的惊人才华。\n情感: 正面\n\n文本: 尽管所有迹象都表明相反，这部糟糕的电影还是设法冒充了一部正式的商业影片，收取全价门票，在电视上大肆宣传，自称能够娱乐小孩和成人。\n情感: 负面\n\n文本: 多年来第一次，德尼罗在情感上有了深刻的挖掘，可能是因为受到了合作伙伴出色表现的启发。\n情感: 正面\n\n文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```  \n许多研究尝试了如何构建上下文示例以最大化性能，并发现**提示格式、训练示例及其顺序的选择可以极大地影响模型的性能**，从几乎是随机猜测到接近最优状态的表现。  \n[Zhao等人（2021年）](https://arxiv.org/abs/2102.09690)研究了少样本分类，并发现几种与LLM（他们使用的是GPT-3）相关的偏见导致了性能的高变异性：（1）*多数标签偏见*，如果示例中的标签分布不平衡；（2）*近期偏见*，模型可能倾向于重复最后出现的标签；（3）*常见Token偏见*，LLM倾向于生成常见的Token而不是罕见的Token。为了克服这些偏见，他们提出了一个方法，即当输入字符串为`N/A`时，调整模型输出的标签概率为均匀。\n\n## 零样本  \n**零样本学习**指的是直接将任务文本输入模型并询问结果。  \n（所有情感分析示例均来源于SST-2）  \n```\n文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```'

In [28]:
context = (RunnableLambda(lambda x: x["question"]) | retriever | format_docs).invoke({"question": question})
print(context)

# 基础提示  
零样本（Zero-shot）和少样本（Few-shot）学习是两种基本的模型提示方法，这两种方法最早由多篇大语言模型论文提出，并广泛用于评估大语言模型的性能。

## 少样本  
**少样本学习**通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。因此，少样本学习通常比零样本学习有更好的表现。但这种方法会消耗更多的Token，并且当输入和输出文本较长时可能触及上下文长度的限制。  
```
文本: （劳伦斯在舞台上）到处跳跃，跳舞，跑步，出汗，擦脸，充分展现了他初次成名时的惊人才华。
情感: 正面

文本: 尽管所有迹象都表明相反，这部糟糕的电影还是设法冒充了一部正式的商业影片，收取全价门票，在电视上大肆宣传，自称能够娱乐小孩和成人。
情感: 负面

文本: 多年来第一次，德尼罗在情感上有了深刻的挖掘，可能是因为受到了合作伙伴出色表现的启发。
情感: 正面

文本: 我敢打赌视频游戏比电影有趣多了。
情感:
```  
许多研究尝试了如何构建上下文示例以最大化性能，并发现**提示格式、训练示例及其顺序的选择可以极大地影响模型的性能**，从几乎是随机猜测到接近最优状态的表现。  
[Zhao等人（2021年）](https://arxiv.org/abs/2102.09690)研究了少样本分类，并发现几种与LLM（他们使用的是GPT-3）相关的偏见导致了性能的高变异性：（1）*多数标签偏见*，如果示例中的标签分布不平衡；（2）*近期偏见*，模型可能倾向于重复最后出现的标签；（3）*常见Token偏见*，LLM倾向于生成常见的Token而不是罕见的Token。为了克服这些偏见，他们提出了一个方法，即当输入字符串为`N/A`时，调整模型输出的标签概率为均匀。

## 零样本  
**零样本学习**指的是直接将任务文本输入模型并询问结果。  
（所有情感分析示例均来源于SST-2）  
```
文本: 我敢打赌视频游戏比电影有趣多了。
情感:
```


In [29]:
context = (RunnableLambda(lambda x: x["question"]) | step_back | retriever | format_docs).invoke({"question": question})
print(context)

# 基础提示  
零样本（Zero-shot）和少样本（Few-shot）学习是两种基本的模型提示方法，这两种方法最早由多篇大语言模型论文提出，并广泛用于评估大语言模型的性能。

文本: 我敢打赌视频游戏比电影有趣多了。
情感:
```  
许多研究尝试了如何构建上下文示例以最大化性能，并发现**提示格式、训练示例及其顺序的选择可以极大地影响模型的性能**，从几乎是随机猜测到接近最优状态的表现。  
[Zhao等人（2021年）](https://arxiv.org/abs/2102.09690)研究了少样本分类，并发现几种与LLM（他们使用的是GPT-3）相关的偏见导致了性能的高变异性：（1）*多数标签偏见*，如果示例中的标签分布不平衡；（2）*近期偏见*，模型可能倾向于重复最后出现的标签；（3）*常见Token偏见*，LLM倾向于生成常见的Token而不是罕见的Token。为了克服这些偏见，他们提出了一个方法，即当输入字符串为`N/A`时，调整模型输出的标签概率为均匀。

[我个人的辛辣观点] 在我看来，一些关于提示工程的论文没必要长达8页，因为这些技巧可以简单地在一句话或几句话内阐明，剩下的大都是基准测试。一个易于使用且共享的基准测试设施对社区更有益。而设置迭代提示或使用外部工具并非易事，让整个研究界接受这一做法也同样困难。

## 零样本  
**零样本学习**指的是直接将任务文本输入模型并询问结果。  
（所有情感分析示例均来源于SST-2）  
```
文本: 我敢打赌视频游戏比电影有趣多了。
情感:
```


### test 4

In [30]:
from langchain_core.runnables import RunnablePassthrough

step_back_and_original = RunnablePassthrough.assign(step_back=step_back)

In [31]:
step_back_and_original.invoke({"question": question})

{'question': '什么是few shot？', 'step_back': 'Few shot是什么意思？'}

In [34]:
# dict -> dict -> str
(step_back_and_original | RunnableLambda(lambda x: x["step_back"])).invoke({"question": question})

'Few shot是什么意思？'

In [35]:
# dict -> dict -> str -> docs
(step_back_and_original | RunnableLambda(lambda x: x["step_back"]) | retriever).invoke({"question": question})

[Document(page_content='# 基础提示  \n零样本（Zero-shot）和少样本（Few-shot）学习是两种基本的模型提示方法，这两种方法最早由多篇大语言模型论文提出，并广泛用于评估大语言模型的性能。', metadata={'Header 1': '基础提示'}),
 Document(page_content='文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```  \n许多研究尝试了如何构建上下文示例以最大化性能，并发现**提示格式、训练示例及其顺序的选择可以极大地影响模型的性能**，从几乎是随机猜测到接近最优状态的表现。  \n[Zhao等人（2021年）](https://arxiv.org/abs/2102.09690)研究了少样本分类，并发现几种与LLM（他们使用的是GPT-3）相关的偏见导致了性能的高变异性：（1）*多数标签偏见*，如果示例中的标签分布不平衡；（2）*近期偏见*，模型可能倾向于重复最后出现的标签；（3）*常见Token偏见*，LLM倾向于生成常见的Token而不是罕见的Token。为了克服这些偏见，他们提出了一个方法，即当输入字符串为`N/A`时，调整模型输出的标签概率为均匀。', metadata={'Header 1': '基础提示', 'Header 2': '少样本'}),
 Document(page_content='[我个人的辛辣观点] 在我看来，一些关于提示工程的论文没必要长达8页，因为这些技巧可以简单地在一句话或几句话内阐明，剩下的大都是基准测试。一个易于使用且共享的基准测试设施对社区更有益。而设置迭代提示或使用外部工具并非易事，让整个研究界接受这一做法也同样困难。', metadata={'Header 1': '提示工程'}),
 Document(page_content='## 零样本  \n**零样本学习**指的是直接将任务文本输入模型并询问结果。  \n（所有情感分析示例均来源于SST-2）  \n```\n文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```', metadata={'Header 1': '基础提示', 'Header 2': '零样本'})]

In [36]:
# Generate
# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{normal_context}"
    "{step_back_context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI()
output_parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "normal_context": (lambda x: x["question"])| retriever | format_docs, 
        "step_back_context": step_back_and_original | RunnableLambda(lambda x: x["step_back"]) | retriever | format_docs, 
        "question": lambda x: x["question"]
    }
    | prompt
    | llm
    | output_parser
)

question = "什么是few shot？"
print(f"Q: {question}\nA: ")

result = rag_chain.invoke({"question": question})
print(result)


Q: 什么是few shot？
A: 
少样本学习（Few-shot learning）是一种模型提示方法，通过提供包含目标任务输入和期望输出的一系列高质量示例，帮助模型更好地理解人类意图和评价标准，从而在少量示例下学习并表现出色。
